# Hotel Booking Data Analysis — Portfolio Edition

A reproducible exploratory analysis of hotel booking patterns, cancellations, guest origins, average daily rates (ADR), and booking segments.

**Source:** Adapted from the repository's original `Hotel_booking_Analysis_Shan_Singh.ipynb`; see that notebook for the original tutorial material. This edition restructures the analysis and adds business interpretation prompts. Results are calculated when the notebook runs; no findings are assumed in advance.

## 1. Load data and check its structure

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
DATA_PATH = Path('hotel_bookings.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / 'hotel_bookings.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError('Place hotel_bookings.csv in the notebook directory or its parent.')
df = pd.read_csv(DATA_PATH)
print(f'Raw rows: {len(df):,} | Columns: {df.shape[1]}')
display(df.head())

In [ ]:
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_pct': df.isna().mean().mul(100).round(2)}).sort_values('missing', ascending=False).head(15))
print('Exact duplicate rows:', df.duplicated().sum())

## 2. Cleaning decisions

The original notebook excludes bookings with zero adults, children, and babies, then removes exact duplicate rows. Here we also treat missing guest counts as zero **only for the invalid-guest check**. The original data remains in `df`. Exact duplicates may sometimes represent separate reservations with identical attributes, so the deduplicated dataset is an analytical choice rather than proof that those reservations are erroneous.

In [ ]:
guest_cols = ['adults', 'children', 'babies']
invalid_guests = df[guest_cols].fillna(0).sum(axis=1).eq(0)
valid = df.loc[~invalid_guests].copy()
clean = valid.drop_duplicates().copy()
print(f'Raw bookings: {len(df):,}')
print(f'Zero-guest rows excluded: {invalid_guests.sum():,}')
print(f'Exact duplicates excluded: {valid.duplicated().sum():,}')
print(f'Rows retained: {len(clean):,}')
print('Cancellation rate on cleaned data: {:.1%}'.format(clean['is_canceled'].mean()))

## 3. Booking cancellations

**Business question:** Do cancellation rates differ by hotel type and by lead time? These are descriptive comparisons, not causal effects.

In [ ]:
hotel_summary = clean.groupby('hotel', dropna=False).agg(bookings=('is_canceled','size'), cancellation_rate=('is_canceled','mean'), median_lead_time=('lead_time','median')).sort_values('bookings',ascending=False)
display(hotel_summary.style.format({'cancellation_rate':'{:.1%}', 'median_lead_time':'{:.0f}'}))
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=clean, x='hotel', y='is_canceled', estimator=np.mean, errorbar=None, ax=ax)
ax.set(title='Cancellation rate by hotel type', ylabel='Cancellation rate', xlabel='Hotel type')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f'{x:.0%}'))
plt.tight_layout(); plt.show()

In [ ]:
lead_bins = [-1, 7, 30, 90, 180, np.inf]
lead_labels = ['0–7 days','8–30 days','31–90 days','91–180 days','181+ days']
clean['lead_time_band'] = pd.cut(clean['lead_time'], bins=lead_bins, labels=lead_labels)
lead_summary = clean.groupby('lead_time_band', observed=True).agg(bookings=('is_canceled','size'), cancellation_rate=('is_canceled','mean'))
display(lead_summary.style.format({'cancellation_rate':'{:.1%}'}))
ax = lead_summary['cancellation_rate'].plot.bar(figsize=(9,4), title='Cancellation rate by booking lead time', rot=0)
ax.set(ylabel='Cancellation rate', xlabel='Lead time'); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f'{x:.0%}'))
plt.tight_layout(); plt.show()

## 4. Booking channels and customer segments

Identify which market segments generate the most reservations, and distinguish volume from cancellation risk.

In [ ]:
segment = clean.groupby('market_segment',dropna=False).agg(bookings=('is_canceled','size'), cancellation_rate=('is_canceled','mean')).sort_values('bookings',ascending=False)
display(segment.head(12).style.format({'cancellation_rate':'{:.1%}'}))
fig, ax = plt.subplots(figsize=(10,5))
segment['bookings'].head(10).sort_values().plot.barh(ax=ax)
ax.set(title='Top market segments by booking count', xlabel='Bookings',ylabel='Market segment')
plt.tight_layout();plt.show()

## 5. ADR and room assignments

ADR means average daily rate. Compare ADR for **non-cancelled bookings** and examine room assignment mismatches. Extreme ADR values can distort means; report medians alongside them.

In [ ]:
stays = clean.loc[clean['is_canceled'].eq(0)].copy()
adr_by_hotel = stays.groupby('hotel')['adr'].agg(['count','mean','median']).round(2)
display(adr_by_hotel)
upper = stays['adr'].quantile(0.99)
plot_adr = stays.loc[stays['adr'].between(0,upper)]
fig, ax = plt.subplots(figsize=(8,4))
sns.boxplot(data=plot_adr, x='hotel', y='adr', ax=ax)
ax.set(title='ADR by hotel (non-cancelled; 0–99th percentile shown)',ylabel='ADR')
plt.tight_layout();plt.show()
print(f'99th percentile ADR used for visualization: {upper:.2f}')

In [ ]:
stays['room_mismatch'] = stays['reserved_room_type'].ne(stays['assigned_room_type'])
room = stays.groupby('hotel').agg(non_cancelled_bookings=('room_mismatch','size'), room_mismatch_rate=('room_mismatch','mean'))
display(room.style.format({'room_mismatch_rate':'{:.1%}'}))
print('Room mismatch is not necessarily an error; upgrades and operational assignments may explain differences.')

## 6. Guest origins and arrival seasonality

Country values are booking-origin codes and may be missing. Arrival month counts are descriptive; do not interpret them as occupancy or demand without accounting for hotel capacity and observation periods.

In [ ]:
origins = stays['country'].fillna('Unknown').value_counts().head(10)
display(origins.rename_axis('country').reset_index(name='non_cancelled_bookings'))
ax = origins.sort_values().plot.barh(figsize=(9,5),title='Top guest origin codes (non-cancelled bookings)')
ax.set(xlabel='Bookings',ylabel='Country code');plt.tight_layout();plt.show()

In [ ]:
month_order = ['January','February','March','April','May','June','July','August','September','October','November','December']
arrivals = stays['arrival_date_month'].value_counts().reindex(month_order,fill_value=0)
ax = arrivals.plot.bar(figsize=(11,4),title='Non-cancelled bookings by arrival month')
ax.set(xlabel='Arrival month',ylabel='Bookings');plt.xticks(rotation=45,ha='right');plt.tight_layout();plt.show()

## 7. Executive summary — generated from observed results

The statements below are computed from the cleaned dataset. They are descriptive associations and should be reviewed before making operational recommendations.

In [ ]:
top_hotel = hotel_summary['bookings'].idxmax()
worst_cancel = hotel_summary['cancellation_rate'].idxmax()
top_segment = segment['bookings'].idxmax()
print(f'• {len(clean):,} rows retained after cleaning; overall cancellation rate {clean.is_canceled.mean():.1%}.')
print(f'• Highest booking volume hotel type: {top_hotel}.')
print(f'• Highest observed cancellation rate hotel type: {worst_cancel} ({hotel_summary.loc[worst_cancel,"cancellation_rate"]:.1%}).')
print(f'• Largest market segment: {top_segment} ({segment.loc[top_segment,"bookings"]:,} bookings).')
print(f'• Highest volume guest origin among non-cancelled bookings: {origins.index[0]} ({origins.iloc[0]:,}).')
print('• Recommendations to investigate: cancellation policies by segment and lead time, seasonal staffing, and room assignment workflows.')

## Limitations and next steps

- This is observational booking data; correlations do not prove causes.
- Deduplicating identical rows may remove legitimate reservations; compare results with and without deduplication.
- `is_canceled == 0` does not independently prove that a guest completed a stay.
- ADR outliers are clipped **only in the boxplot**, not removed from the analysis.
- Results may depend on missing-value conventions and the date coverage of the dataset.
- Future work: sensitivity analysis, cancellation forecasting with leakage checks, and an interactive dashboard.